Secrets, connection strings, API keys, certificates, and encryption keys are the most sensitive assets in any cloud application. Azure provides two complementary services to manage them securely:

- **Azure Key Vault** — a managed HSM-backed store for secrets, keys, and certificates. Applications retrieve credentials at runtime rather than embedding them in code or config files.
- **Managed Identities** — Azure-managed service principals that give Azure resources (VMs, App Services, AKS pods) an identity in Entra ID — so they can authenticate to Key Vault and other services **without any stored credentials**.

Together they eliminate the fundamental credential management problem: *how does an application authenticate without storing a password somewhere?*

## Azure Key Vault

### Vault tiers

| Tier | Backing | Use case |
|---|---|---|
| **Standard** | Software-protected | General purpose |
| **Premium** | HSM-protected (FIPS 140-2 Level 2) | Regulated industries, compliance |
| **Managed HSM** | Dedicated single-tenant HSM (FIPS 140-2 Level 3) | Highest assurance, full key ownership |

### The three object types

#### Secrets

**Secrets** store any sensitive string value — database connection strings, API keys, passwords, storage account keys.

- Versioned — each update creates a new version; previous versions remain accessible
- Optional **activation date** and **expiry date** per version
- Soft-delete enabled by default — deleted secrets are retained for 7–90 days and can be recovered

#### Keys

**Keys** are cryptographic keys used for encrypt/decrypt and sign/verify operations. The key material **never leaves Key Vault** — applications call the Key Vault API to use the key.

| Key type | Description |
|---|---|
| **RSA** | 2048, 3072, 4096-bit asymmetric |
| **EC** | P-256, P-384, P-521 elliptic curve |
| **Symmetric (oct)** | AES — HSM and Managed HSM only |

Key operations: encrypt, decrypt, sign, verify, wrapKey, unwrapKey.

Use cases: customer-managed keys (CMK) for Azure Storage TDE, Cosmos DB, SQL TDE, disk encryption.

#### Certificates

**Certificates** store X.509 TLS/SSL certificates. Key Vault can:
- Generate a self-signed certificate
- Issue a certificate from a **CA integration** (DigiCert, GlobalSign) — Key Vault creates the CSR, submits it, and imports the signed cert automatically
- Import an existing certificate (PFX/PEM)
- **Auto-renew** certificates before expiry

When a certificate is stored in Key Vault, the private key is also stored as a Key object — accessible separately if needed.

### Access policies vs RBAC

Key Vault supports two permission models:

| Model | Description | Recommended? |
|---|---|---|
| **Vault access policy** | Legacy — coarse-grained permissions per vault; one policy per identity | No |
| **Azure RBAC** | Fine-grained roles per object type; integrates with Entra ID RBAC | ✅ Yes |

Built-in RBAC roles for Key Vault:

| Role | Access |
|---|---|
| **Key Vault Administrator** | Full management of all object types + vault settings |
| **Key Vault Secrets Officer** | Create, read, update, delete secrets |
| **Key Vault Secrets User** | Read secrets only |
| **Key Vault Keys Officer** | Create, read, update, delete keys |
| **Key Vault Crypto User** | Use keys for crypto operations (encrypt/decrypt/sign) |
| **Key Vault Certificate Officer** | Manage certificates |
| **Key Vault Reader** | Read vault metadata only (not secret/key values) |

### Networking

| Option | Description |
|---|---|
| **Public + firewall** | Allow specific IP ranges or VNet service endpoints |
| **Private endpoint** | Private IP in VNet — `privatelink.vaultcore.azure.net`; no public exposure |

Always use private endpoints for production vaults.

### Soft-delete and purge protection

| Feature | Description |
|---|---|
| **Soft-delete** | Deleted objects retained for 7–90 days; recoverable; enabled by default |
| **Purge protection** | When enabled, objects in soft-delete state cannot be purged before the retention period — even by admins; required for CMK scenarios |

### Key Vault logging

Enable the `AuditEvent` diagnostic log to record every access to secrets, keys, and certificates — who accessed what, when, from which IP. Send to Log Analytics for querying and alerting on suspicious access patterns.

## Managed Identities

A **Managed Identity** is an automatically managed service principal in Entra ID, assigned to an Azure resource. The resource uses this identity to authenticate to any service that supports Entra ID authentication — without storing any credentials.

Azure handles the identity lifecycle: creation, credential rotation, and deletion. You never see or manage the underlying certificate/secret.

### System-assigned vs user-assigned

| | System-assigned | User-assigned |
|---|---|---|
| **Lifecycle** | Tied to the resource — deleted when the resource is deleted | Independent resource — persists independently |
| **Sharing** | One identity per resource; cannot be shared | Can be assigned to multiple resources |
| **Use case** | Single-resource workloads | Multiple VMs sharing the same identity; pre-provisioning |

### How it works

```
1. Enable Managed Identity on resource (VM, App Service, Function, AKS workload)
2. Azure creates a service principal in Entra ID for the resource
3. Grant the identity RBAC permissions on target services (Key Vault, Storage, SQL, etc.)
4. Application calls Azure Instance Metadata Service (IMDS) at 169.254.169.254
   to get a short-lived Entra ID token — automatically refreshed
5. Application uses token to call the target service API
```

The **DefaultAzureCredential** class in the Azure SDK handles step 4 automatically — it tries multiple authentication mechanisms in order:

```
DefaultAzureCredential tries (in order):
  1. EnvironmentCredential        (env vars: AZURE_CLIENT_ID, AZURE_CLIENT_SECRET, etc.)
  2. WorkloadIdentityCredential   (AKS workload identity)
  3. ManagedIdentityCredential    (VM / App Service / Function Managed Identity)
  4. AzureCliCredential           (local development: az login)
  5. AzurePowerShellCredential    (local development: Connect-AzAccount)
  6. AzureDeveloperCliCredential  (azd auth login)
```

This means the same application code works in production (Managed Identity) and locally (Azure CLI credentials) without any changes.

### Workload Identity for AKS

Pods in AKS cannot use VM-level Managed Identity directly. **Workload Identity** (the recommended approach) uses the Kubernetes Service Account token projected into the pod and exchanges it for an Entra ID token via OIDC federation:

```
Pod (Kubernetes Service Account token)
  → Workload Identity webhook injects AZURE_CLIENT_ID + token file
  → WorkloadIdentityCredential exchanges token for Entra ID token
  → Pod calls Key Vault / Storage / etc.
```

No secrets are stored in Kubernetes Secrets or pod environment variables.

## Secrets Store CSI Driver (AKS)

The **Secrets Store CSI Driver** with the Azure Key Vault provider mounts Key Vault secrets, keys, and certificates directly as files or environment variables in AKS pods — using Workload Identity for authentication:

```yaml
# SecretProviderClass — maps Key Vault objects to Kubernetes volume
apiVersion: secrets-store.csi.x-k8s.io/v1
kind: SecretProviderClass
metadata:
  name: my-kv-provider
spec:
  provider: azure
  parameters:
    usePodIdentity: "false"
    clientID: "<user-assigned-managed-identity-client-id>"
    keyvaultName: "my-keyvault"
    objects: |
      array:
        - |
          objectName: db-connection-string
          objectType: secret
        - |
          objectName: tls-cert
          objectType: cert
```

The secret appears as a file at a mount path inside the pod — no K8s Secret object required, and the value is never stored in etcd.

## Key Vault Best Practices

### Naming and organisation

- Separate vaults by environment (dev / staging / prod) and sensitivity level
- One vault per application or team — limits blast radius if a vault is misconfigured
- Use naming conventions for secrets: `<app>-<env>-<purpose>` e.g. `orders-prod-db-connection`

### Secret rotation

Automate secret rotation using:
1. **Key Vault + Event Grid** — Key Vault emits `SecretNearExpiry` and `SecretExpired` events to Event Grid
2. **Azure Function** triggered by the event — generates a new credential and updates the Key Vault secret
3. **Application** references the latest version (no version in the URI) — picks up the new secret automatically

```
Key Vault: secret nearing expiry
  → Event Grid: SecretNearExpiry event
    → Azure Function: rotate credential in DB/API + update Key Vault secret
      → App: reads latest secret version on next request
```

### Secret references in App Service and Azure Functions

App Service and Azure Functions support **Key Vault references** in app settings — the runtime resolves the value from Key Vault at startup:

```
App Setting value:  @Microsoft.KeyVault(SecretUri=https://my-kv.vault.azure.net/secrets/db-conn/)
Runtime resolves → actual secret value injected as environment variable
```

The App Service's Managed Identity must have the `Key Vault Secrets User` role on the vault.

## Working with Key Vault and Managed Identities via Python

In [ ]:
# pip install azure-identity azure-keyvault-secrets azure-keyvault-keys azure-keyvault-certificates azure-mgmt-keyvault

In [ ]:
from azure.identity import DefaultAzureCredential
from azure.keyvault.secrets import SecretClient
from azure.keyvault.keys import KeyClient, KeyType
from azure.keyvault.certificates import CertificateClient, CertificatePolicy
from azure.mgmt.keyvault import KeyVaultManagementClient
from azure.mgmt.keyvault.models import VaultCreateOrUpdateParameters, VaultProperties, Sku

credential      = DefaultAzureCredential()   # works locally (az login) and in Azure (Managed Identity)
subscription_id = "<your-subscription-id>"
resource_group  = "my-rg"
vault_url       = "https://my-keyvault.vault.azure.net"

secret_client  = SecretClient(vault_url=vault_url, credential=credential)
key_client     = KeyClient(vault_url=vault_url, credential=credential)
cert_client    = CertificateClient(vault_url=vault_url, credential=credential)
kv_mgmt        = KeyVaultManagementClient(credential, subscription_id)

In [ ]:
import uuid

# Create a Key Vault with RBAC permission model and soft-delete
tenant_id = "<your-tenant-id>"

poller = kv_mgmt.vaults.begin_create_or_update(
    resource_group, "my-keyvault",
    VaultCreateOrUpdateParameters(
        location="eastus",
        properties=VaultProperties(
            tenant_id=tenant_id,
            sku=Sku(family="A", name="standard"),
            enable_rbac_authorization=True,   # use RBAC, not access policies
            soft_delete_retention_in_days=90,
            enable_purge_protection=True,     # prevent purging during retention period
            enable_soft_delete=True
        )
    )
)
vault = poller.result()
print(f"Vault: {vault.name}  URI: {vault.properties.vault_uri}")

In [ ]:
from datetime import datetime, timezone, timedelta

# Set a secret
secret = secret_client.set_secret(
    "db-connection-string",
    "Server=my-sql-server.database.windows.net;Database=mydb;...",
    expires_on=datetime.now(timezone.utc) + timedelta(days=365)
)
print(f"Secret set: {secret.name}  version: {secret.properties.version}")

# Get latest version (no version in name — always returns current)
retrieved = secret_client.get_secret("db-connection-string")
print(f"Value length: {len(retrieved.value)} chars  expires: {retrieved.properties.expires_on}")

# Get a specific version
versioned = secret_client.get_secret("db-connection-string", version=secret.properties.version)
print(f"Specific version: {versioned.properties.version}")

# List all secrets in the vault
for s in secret_client.list_properties_of_secrets():
    print(f"  {s.name:<40} enabled: {s.enabled}  expires: {s.expires_on}")

# List versions of a specific secret
for v in secret_client.list_properties_of_secret_versions("db-connection-string"):
    print(f"  version: {v.version}  created: {v.created_on}  enabled: {v.enabled}")

In [ ]:
# Create an RSA key
key = key_client.create_rsa_key("my-rsa-key", size=2048)
print(f"Key created: {key.name}  type: {key.key_type}")

# Create an EC key
ec_key = key_client.create_ec_key("my-ec-key", curve="P-256")
print(f"EC key created: {ec_key.name}")

# Use a key for cryptographic operations (key never leaves Key Vault)
from azure.keyvault.keys.crypto import CryptographyClient, EncryptionAlgorithm, SignatureAlgorithm
import hashlib

crypto_client = CryptographyClient(key, credential=credential)

# Encrypt and decrypt
plaintext = b"sensitive data"
encrypt_result = crypto_client.encrypt(EncryptionAlgorithm.rsa_oaep, plaintext)
print(f"Encrypted: {len(encrypt_result.ciphertext)} bytes")

decrypt_result = crypto_client.decrypt(EncryptionAlgorithm.rsa_oaep, encrypt_result.ciphertext)
print(f"Decrypted: {decrypt_result.plaintext}")

# Sign and verify a digest
digest = hashlib.sha256(b"document content").digest()
sign_result = crypto_client.sign(SignatureAlgorithm.rs256, digest)
verify_result = crypto_client.verify(SignatureAlgorithm.rs256, digest, sign_result.signature)
print(f"Signature valid: {verify_result.is_valid}")

In [ ]:
# Create a self-signed certificate with auto-renewal
policy = CertificatePolicy(
    issuer_name="Self",
    subject="CN=my-service.contoso.com",
    validity_in_months=12,
    san_dns_names=["my-service.contoso.com", "www.contoso.com"],
    exportable=True,
    key_size=2048,
    # Auto-renew 30 days before expiry
    lifetime_actions=[{"action": {"action_type": "AutoRenew"}, "trigger": {"days_before_expiry": 30}}]
)

poller = cert_client.begin_create_certificate("my-tls-cert", policy)
cert = poller.result()
print(f"Certificate: {cert.name}  expires: {cert.properties.expires_on}")

# List all certificates
for c in cert_client.list_properties_of_certificates():
    print(f"  {c.name:<30} expires: {c.expires_on}  enabled: {c.enabled}")

In [ ]:
from azure.mgmt.msi import ManagedServiceIdentityClient
from azure.mgmt.msi.models import Identity

msi = ManagedServiceIdentityClient(credential, subscription_id)

# Create a user-assigned managed identity
identity = msi.user_assigned_identities.create_or_update(
    resource_group, "my-app-identity",
    Identity(location="eastus")
)
print(f"Identity: {identity.name}")
print(f"  Client ID:    {identity.client_id}")
print(f"  Principal ID: {identity.principal_id}")
print(f"  Tenant ID:    {identity.tenant_id}")

# The principal_id is used to assign RBAC roles
# e.g. grant Key Vault Secrets User on the vault:
# az role assignment create \
#   --role "Key Vault Secrets User" \
#   --assignee <principal_id> \
#   --scope /subscriptions/.../vaults/my-keyvault

In [ ]:
# Demonstrate Key Vault reference pattern used in App Service settings
# The actual resolution happens at the App Service runtime — this shows the reference format

vault_name  = "my-keyvault"
secret_name = "db-connection-string"

# Latest version reference (resolves to current version automatically)
kv_ref_latest = f"@Microsoft.KeyVault(SecretUri=https://{vault_name}.vault.azure.net/secrets/{secret_name}/)"

# Named secret reference using VaultName + SecretName syntax
kv_ref_named = f"@Microsoft.KeyVault(VaultName={vault_name};SecretName={secret_name})"

print("App Service setting value (latest):")
print(f"  {kv_ref_latest}")
print("App Service setting value (named):")
print(f"  {kv_ref_named}")
print("\nSet this as an App Setting value — App Service resolves it at startup via Managed Identity")

## Summary

| Concept | Key Takeaway |
|---|---|
| Key Vault | Managed store for secrets, keys, certificates — HSM-backed at Premium tier |
| Secrets | Versioned strings — connection strings, API keys, passwords |
| Keys | Cryptographic keys — never leave Key Vault; used via API calls |
| Certificates | X.509 certs with auto-renewal; CA integration for automatic issuance |
| RBAC model | Recommended over access policies — fine-grained, Entra ID integrated |
| Key Vault Secrets User | Read-only secret access — assign to application identities |
| Soft-delete | Deleted objects retained 7–90 days; recoverable |
| Purge protection | Prevents purging during retention period — required for CMK |
| Private endpoint | Private IP in VNet — no public exposure; always use in production |
| AuditEvent log | Every access logged — send to Log Analytics for anomaly detection |
| Managed Identity | Azure-managed service principal — no stored credentials |
| System-assigned | Tied to resource lifecycle; one per resource; auto-deleted |
| User-assigned | Independent resource; assignable to multiple resources |
| DefaultAzureCredential | SDK class — tries Managed Identity in Azure, Azure CLI locally |
| Workload Identity | AKS pod identity via OIDC federation — replaces Pod Identity |
| Secrets Store CSI | Mount Key Vault objects as files in AKS pods — no K8s Secrets in etcd |
| Key Vault reference | `@Microsoft.KeyVault(...)` in App Service settings — resolved at startup |
| Secret rotation | Event Grid SecretNearExpiry → Function → update secret → app reads latest |
| CMK pattern | Cosmos DB / SQL / Storage TDE — your Key Vault key controls encryption |
| Managed HSM | Dedicated single-tenant HSM — highest assurance, full key ownership |